In [ ]:
# Ubuntu 24.04 system libraries required by headless Chromium.
!sudo apt-get update -y
!sudo apt-get install -y libnss3 libnspr4 libatk1.0-0t64 libatk-bridge2.0-0t64 libatspi2.0-0t64 libcups2t64 libasound2t64 libgtk-3-0t64 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libdrm2 libx11-xcb1 libxcb1 libxcb-dri3-0 libxshmfence1 libpango-1.0-0 libcairo2 libxrender1 libfontconfig1 libfreetype6 libglib2.0-0t64 libdbus-1-3 libexpat1 libxext6 libx11-6 libegl1 fonts-liberation


# USMAN AND UMAR — Colab Backend with Cloudflare Quick Tunnel

This notebook runs the OpenSwarm multi-agent executor in Google Colab and exposes it through a temporary Cloudflare Quick Tunnel. `/execute` remains protected by `Authorization: Bearer <APP_CLIENT_TOKEN>`.

## One-time setup

1. In Colab, add `APP_CLIENT_TOKEN` in the **Secrets** panel and allow notebook access. Add at least one provider secret: `NVIDIA_API_KEY` (preferred) or `OPENROUTER_API_KEY` (fallback). NVIDIA takes priority if both are present.
2. NVIDIA is pinned to `nvidia/nemotron-3.5-bolt-e4b`. Optionally add `OPENROUTER_MODEL` (default `openrouter/free`) as a Colab Secret.
3. Run all cells from top to bottom. The last tunnel cell prints a temporary `https://....trycloudflare.com` URL. Copy that URL into the Android app's **Settings → Cloud Servers**. The app adds `/execute` automatically.
4. Keep the Colab runtime and tunnel cell alive while using cloud tasks. Quick Tunnel URLs are temporary and can change whenever the tunnel or runtime restarts; update the app URL if that happens.

Cloud tasks must include at least one public `http(s)` URL in the prompt. Browser Use is restricted to those hosts and scoped click/type/submit actions limited to the exact user request. FMHY tasks dynamically add one matching external host. Do not include secrets or private information. Full server tracebacks are logged to `/tmp/usman-fastapi.log`; the public API returns a credential-safe error summary.


## 1. Install Cloudflare Tunnel client

Downloads the official Linux AMD64 `cloudflared` binary. Colab Linux runtimes are expected to use AMD64.


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version


## 2. Load the FastAPI OpenSwarm multi-agent executor

The notebook clones this branch's `cloud-backend/` folder, installs FastAPI and Browser Use, and installs Chromium for scoped browser execution.


In [ ]:
# Install Ubuntu 24.04 Chromium runtime libraries before Playwright setup.
!sudo apt-get update
!sudo apt-get install -y libnss3 libnspr4 libasound2t64 libatk1.0-0t64 libatk-bridge2.0-0t64 libcups2t64 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libatspi2.0-0t64


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/saedhawwa23-rgb/usman-and-umar-app.git"
BRANCH = "feature/hybrid-task-orchestrator"
PROJECT_DIR = Path("/content/usman-and-umar-app")

if not (PROJECT_DIR / ".git").exists():
    if PROJECT_DIR.exists():
        raise RuntimeError(
            f"{PROJECT_DIR} exists but is not a Git checkout. Move it aside and rerun this cell."
        )
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        cwd=PROJECT_DIR,
        check=True,
    )

BACKEND_DIR = PROJECT_DIR / "cloud-backend"
if not (BACKEND_DIR / "main.py").is_file() or not (BACKEND_DIR / "browser_use_interactive.py").is_file():
    raise RuntimeError("The expected FastAPI or Browser Use source file is missing from the checkout.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(BACKEND_DIR / "requirements.txt")],
    check=True,
)
PLAYWRIGHT_BROWSERS_PATH = "/content/.cache/ms-playwright"
os.environ["PLAYWRIGHT_BROWSERS_PATH"] = PLAYWRIGHT_BROWSERS_PATH
os.makedirs(PLAYWRIGHT_BROWSERS_PATH, exist_ok=True)
PLAYWRIGHT_FAILURE_MARKER = Path("/tmp/usman_playwright_install_failed")
PLAYWRIGHT_FAILURE_MARKER.unlink(missing_ok=True)

playwright_install_command = [sys.executable, "-m", "playwright", "install", "chromium"]
try:
    subprocess.run(playwright_install_command, check=True)
    print("Playwright Chromium install completed through the Python subprocess.")
except subprocess.CalledProcessError as exc:
    print(f"Python Playwright install failed (exit {exc.returncode}); the next bash cell will retry directly.")
    PLAYWRIGHT_FAILURE_MARKER.touch()
print("Backend and Browser Use dependencies are ready.")


In [ ]:
%%bash
set -euo pipefail
export PLAYWRIGHT_BROWSERS_PATH="/content/.cache/ms-playwright"
mkdir -p "$PLAYWRIGHT_BROWSERS_PATH"
if [[ -f /tmp/usman_playwright_install_failed ]]; then
  echo "Retrying Playwright install directly in bash..."
  pip install -q playwright
  playwright install chromium
  rm -f /tmp/usman_playwright_install_failed
  echo "Playwright Chromium install succeeded through the bash fallback."
else
  echo "Python Playwright install succeeded; bash fallback not needed."
fi


## 3. Read Colab Secrets (values are never printed)

Required: `APP_CLIENT_TOKEN` and at least one inference key. If `NVIDIA_API_KEY` is present, the backend always uses `nvidia/nemotron-3.5-bolt-e4b` at NVIDIA's OpenAI-compatible endpoint; otherwise it falls back to `OPENROUTER_API_KEY`.


In [ ]:
import os
from google.colab import userdata

def read_secret(name, *, required=False):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    value = (value or "").strip()
    if required and not value:
        raise RuntimeError(
            f"Missing Colab Secret {name}. Add it in the Secrets panel and enable notebook access."
        )
    return value

APP_CLIENT_TOKEN = read_secret("APP_CLIENT_TOKEN", required=True)
NVIDIA_API_KEY = read_secret("NVIDIA_API_KEY")
OPENROUTER_API_KEY = read_secret("OPENROUTER_API_KEY")
OPENROUTER_MODEL = read_secret("OPENROUTER_MODEL") or os.environ.get("OPENROUTER_MODEL", "").strip() or "openrouter/free"

if not NVIDIA_API_KEY and not OPENROUTER_API_KEY:
    raise RuntimeError("Add NVIDIA_API_KEY or OPENROUTER_API_KEY in Colab Secrets; NVIDIA is preferred when both are set.")

os.environ["APP_CLIENT_TOKEN"] = APP_CLIENT_TOKEN
if NVIDIA_API_KEY:
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
if OPENROUTER_API_KEY:
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENROUTER_MODEL"] = OPENROUTER_MODEL

selected_provider = "NVIDIA" if NVIDIA_API_KEY else "OpenRouter"
print(f"Secrets loaded; selected inference provider: {selected_provider}. Key values are hidden.")


## 4. Start the FastAPI server on port 7860

This starts the real OpenSwarm Manager Agent locally in Colab. It delegates to Browser Agent or Coder Agent and loops to a final answer. It does not forward tasks to an external OpenSwarm URL.


In [ ]:
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

API_BASE = "http://127.0.0.1:7860"

def local_health_ok():
    try:
        with urllib.request.urlopen(f"{API_BASE}/health", timeout=2) as response:
            return response.status == 200
    except Exception:
        return False

if local_health_ok():
    print("FastAPI is already healthy on port 7860.")
else:
    server_log_path = Path("/tmp/usman-fastapi.log")
    server_log = open(server_log_path, "a", encoding="utf-8")
    fastapi_process = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"],
        cwd=BACKEND_DIR,
        env=os.environ.copy(),
        stdout=server_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(30):
        if local_health_ok():
            break
        if fastapi_process.poll() is not None:
            break
        time.sleep(1)
    if not local_health_ok():
        server_log.flush()
        server_log.close()
        log_text = server_log_path.read_text(encoding="utf-8", errors="replace")[-3000:]
        raise RuntimeError(f"FastAPI did not become healthy. Recent server log:\n{log_text}")
    print("FastAPI health check passed at /health.")


## 5. Start Cloudflare Quick Tunnel and copy its temporary URL

Run this after the FastAPI health check succeeds. Keep this cell running. Copy the printed `trycloudflare.com` URL into the Android app's Server 1 URL setting; do not append `/execute`.


In [ ]:
import queue
import re
import subprocess
import threading

if "cloudflared_process" in globals() and cloudflared_process.poll() is None:
    cloudflared_process.terminate()
    try:
        cloudflared_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        cloudflared_process.kill()
        cloudflared_process.wait(timeout=5)

print("Starting Cloudflare Quick Tunnel...")
cloudflared_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
cloudflare_url_queue = queue.Queue(maxsize=1)

def _read_cloudflared_logs():
    if cloudflared_process.stdout is None:
        return
    for line in cloudflared_process.stdout:
        print(line, end="", flush=True)
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match and cloudflare_url_queue.empty():
            cloudflare_url_queue.put(match.group(0))

cloudflared_log_reader = threading.Thread(target=_read_cloudflared_logs, daemon=True)
cloudflared_log_reader.start()
try:
    cloudflare_public_url = cloudflare_url_queue.get(timeout=120)
except queue.Empty as exc:
    if cloudflared_process.poll() is not None:
        raise RuntimeError(f"cloudflared exited with code {cloudflared_process.returncode} before printing a URL.") from exc
    raise TimeoutError("Cloudflare Quick Tunnel did not print a trycloudflare.com URL within 120 seconds.") from exc

print(f"\nYOUR PUBLIC URL IS: {cloudflare_public_url}")
print("Copy this URL into the Android app's Cloud Servers settings (base URL only; /execute is appended automatically).")
print("This URL is temporary and may change after the tunnel or Colab runtime restarts.")


## 6. Optional: test the Browser Use helper interactively

The helper restricts browsing to the hostnames you explicitly allow and runs scoped tasks. It uses `nvidia/nemotron-3.5-bolt-e4b` with `NVIDIA_API_KEY`, then falls back to `OPENROUTER_API_KEY`.


In [ ]:
import sys
sys.path.insert(0, str(BACKEND_DIR))
from browser_use_interactive import run_browser_task

result = await run_browser_task(
    "Read the Python installation instructions and summarize the requirements.",
    allowed_domains=["docs.python.org"],
    max_steps=15,
)
print(result)
